In [ ]:
from glob import glob
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

***

### Loading and formatting data

In [ ]:
paths = glob(r'E:\Mara BUD analyses\*\*ET*\output\*ET*')

In [ ]:
all_dfs = []

for path in paths:

    path = Path(path)

    ASH1CLB2_df = pd.read_csv(path / 'ASH1CLB2.csv')
    SRL1_df     = pd.read_csv(path / 'SRL1.csv')
    SUN4_df     = pd.read_csv(path / 'SUN4.csv')

    strain = path.parts[-3].split(' ')[0].upper()
    replicate = path.parts[-3].split(' ')[1]

    for mRNA, df_tmp in zip(
        ['ASH1CLB2', 'SRL1', 'SUN4'],
        [ASH1CLB2_df, SRL1_df, SUN4_df]
    ):
        df = df_tmp.copy()  # important
        p = Path(path)
        new_path = '../' + '..'+ '/' + '/'.join(p.parts[2:])  # skip the 'E:\\' part

        df['source_folder'] = new_path
        df['strain'] = strain
        df['replicate'] = replicate
        df['mRNA'] = mRNA

        all_dfs.append(df)

# concatenate everything
df = pd.concat(all_dfs, ignore_index=True)

In [ ]:
strain_to_condition = {
    "YET918": "ΔSUN4",
    "YET563": "ΔSHE2",
    "YET572": "ΔSHE3",
    "YET915": "WT",
    "YET916": "ΔSSD1",
    "YET919": "ΔMPT5",
    "YET921": "ΔSEC72"
}
df["condition"] = df["strain"].map(strain_to_condition)
df["condition + strain"] = df["condition"] + ' ' +  df["strain"]
df = df.drop(['nuclei_bud','nuclei'],axis=1)
# Although very little, there was some overdetection in the SUN4 channel for the deletion strains, resulting in spots where there shouldn't be. 
# Feel free to comment out this code to test the effect.
columns_to_change=['spots', 'dense_regions', 'decomposed_RNAs','tx', 'nascent_RNAs', 'total_RNAs', 'spots_bud', 'dense_regions_bud',
 'decomposed_RNAs_bud', 'tx_bud', 'nascent_RNAs_bud','total_RNAs_bud', 'spots_mother', 'total_RNAs_mother']
mask = (df['condition']=='ΔSUN4') & (df['mRNA']=='SUN4')
df.loc[mask, columns_to_change] = 0
df.to_csv('Mara_budSegmentationData20260311.csv') 

***

### Plotting all data

In [ ]:
df['fov']=df['source_folder'].str[-2:]
df['uid'] = (
    df['strain'].astype(str) + '_' +
    df['replicate'].astype(str) + '_' +
    df['image_cell_id'].astype(str) + '_' +
    df['fov'].astype(str)
)
df.to_csv('Mara_budSegmentationData20260311.csv') 

In [ ]:
df_buds = df[df['area_bud']>0]

sns.set(style="whitegrid")
# Melt mother/bud spots into long format per cell
plot_df = df_buds.melt(
    id_vars=['strain', 'condition', 'condition + strain', 'replicate', 'mRNA', 'uid'],
    value_vars=['total_RNAs_mother', 'total_RNAs_bud'],
    var_name='cell_type',
    value_name='spots_count'
)

# Rename for nicer axis labels
plot_df['cell_type'] = plot_df['cell_type'].replace({
    'total_RNAs_mother': 'Mother',
    'total_RNAs_bud': 'Bud'
})

# Create a color palette for replicates
replicates = sorted(plot_df['replicate'].unique())
palette = sns.color_palette("tab10", len(replicates))
replicate_colors = dict(zip(replicates, palette))

# Compute cell counts per column (condition + strain)
cell_counts = plot_df.groupby('condition + strain')['uid'].nunique()
# Append counts to column names
plot_df['condition + strain count'] = plot_df['condition + strain'].apply(lambda x: f"{x} (N={cell_counts[x]})")

# Function to plot lines for each cell in the current facet
def plot_cell_lines(data, **kwargs):
    for cell_id, cell_df in data.groupby('uid'):
        rep = cell_df['replicate'].iloc[0]
        plt.plot(
            cell_df['cell_type'],
            cell_df['spots_count'],
            color=replicate_colors[rep],
            alpha=0.7,
            linewidth=0.2
        )

# Create FacetGrid: rows = mRNA, cols = condition + strain with counts
g = sns.FacetGrid(
    plot_df,
    row='mRNA',
    col='condition + strain count',
    sharey=True,
    sharex=True,
    height=3,
    aspect=1.2,
    margin_titles=True
)

# Plot lines
g.map_dataframe(plot_cell_lines)

# Axis labels and titles
g.set_axis_labels("Cell type", "Total RNAs")
g.set_titles(row_template="{row_name}", col_template="{col_name}")

# Legend for replicates
handles = [mlines.Line2D([], [], color=replicate_colors[r], label=f"rep {r}") for r in replicates]
g.axes[0, -1].legend(
    handles=handles,
    title="Replicate",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.savefig('total_RNAs_mothers_vs_buds.png',dpi=600)
plt.show()